# Pulling Data from FRED


In [1]:
import os
import pandas as pd
from pandas_datareader import data as web

In [2]:
START_DATE = "1984-01-01"
END_DATE = "2014-12-31"

In [3]:
SERIES = {
    "CPI":     "CPIAUCSL",          # CPI, all items, SA
    "CoreCPI": "USACPICORMINMEI",   # CPI excluding food & energy
    "PCE":     "PCE",               # Personal consumption expenditures
    "CorePCE": "PCEPILFE",   # As listed in paper (real PCE % change)
    "IP":      "INDPRO",            # Industrial production index
    "UNEM":    "UNRATE",            # Civilian unemployment rate
    "INC":     "DPCERL1M225NBEA",   # Real PCE % change (same code as CorePCE in paper)
    "WORK":    "PAYEMS",            # All employees, total nonfarm
    "HS":      "HOUST",             # Housing starts
    "GS5":     "GS5",               # 5-year Treasury constant maturity rate
    "TB3MS":   "TB3MS",             # 3-month Treasury bill, secondary market
}

CORE_PCE_ALT_CODE = "PCEPILFE"  # standard core PCE deflator, for comparison

In [4]:
def fetch_series(name, code):
    print(f"Fetching {name} ({code})...")
    s = web.DataReader(code, "fred", START_DATE, END_DATE)
    s.columns = [name]
    return s


In [5]:
frames = []
for name, code in SERIES.items():
    frames.append(fetch_series(name, code))

frames.append(fetch_series("CorePCE_alt_PCEPILFE", CORE_PCE_ALT_CODE))

Fetching CPI (CPIAUCSL)...
Fetching CoreCPI (USACPICORMINMEI)...
Fetching PCE (PCE)...
Fetching CorePCE (PCEPILFE)...
Fetching IP (INDPRO)...
Fetching UNEM (UNRATE)...
Fetching INC (DPCERL1M225NBEA)...
Fetching WORK (PAYEMS)...
Fetching HS (HOUST)...
Fetching GS5 (GS5)...
Fetching TB3MS (TB3MS)...
Fetching CorePCE_alt_PCEPILFE (PCEPILFE)...


In [6]:
df = pd.concat(frames, axis=1)

# Compute SPREAD = 5yr Treasury - 3-month T-bill
df["SPREAD"] = df["GS5"] - df["TB3MS"]

# Align to monthly frequency, forward-fill small gaps from mixed release schedules
df = df.asfreq("MS")
df = df.ffill()
df = df.loc[START_DATE:END_DATE]

In [7]:
OUT_PATH = "/Users/kartikmisra/Documents/ECO723 Project/inflation_forecasting_data.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH)